In [1]:
# Plot metrics
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss plot
axes[0].plot(train_losses, label="Train Loss", linewidth=2)
axes[0].plot(val_losses, label="Val Loss", linewidth=2)
axes[0].set_xlabel("Epoch", fontsize=12)
axes[0].set_ylabel("Loss", fontsize=12)
axes[0].set_title("Training and Validation Loss", fontsize=14)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Accuracy plot
axes[1].plot(train_accs, label="Train Accuracy", linewidth=2)
axes[1].plot(val_accs, label="Val Accuracy", linewidth=2)
axes[1].set_xlabel("Epoch", fontsize=12)
axes[1].set_ylabel("Accuracy (%)", fontsize=12)
axes[1].set_title("Training and Validation Accuracy", fontsize=14)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print final metrics
print(f"Final train loss: {train_losses[-1]:.4f} | train accuracy: {train_accs[-1]:.2f}%")
print(f"Final val loss: {val_losses[-1]:.4f} | val accuracy: {val_accs[-1]:.2f}%")
print(f"Best val accuracy: {best_acc:.2f}%")

NameError: name 'plt' is not defined

## Visualization

Plot training and validation metrics over epochs.

In [ ]:
# Training loop with history tracking
train_losses = []
train_accs = []
val_losses = []
val_accs = []
best_acc = 0.0

print("Starting training...\n")

for epoch in range(epochs):
    # Train
    train_loss, train_acc = train_epoch(train_loader, model, criterion, optimizer, device)
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    
    # Validate
    val_loss, val_acc = validate(val_loader, model, criterion, device)
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    
    # Learning rate step
    scheduler.step()
    
    # Print progress
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch [{epoch + 1}/{epochs}]")
        print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
        print(f"  Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.2f}%")
        print()
    
    # Save checkpoint if best
    if val_acc > best_acc:
        best_acc = val_acc
        checkpoint = {
            "epoch": epoch,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "best_acc": best_acc,
        }
        torch.save(checkpoint, os.path.join(checkpoint_dir, "model_best.pth"))

print(f"Training complete! Best validation accuracy: {best_acc:.2f}%")

## Section 8: Execute Training and Monitor Progress

Run the complete training loop for multiple epochs, track metrics, save checkpoints, and visualize training progress.

In [ ]:
def validate(loader, model, criterion, device):
    model.eval()
    loss_meter = AverageMeter()
    acc_meter = AverageMeter()
    
    with torch.no_grad():
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            
            # Forward pass
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            # Metrics
            acc = accuracy(outputs, targets)
            loss_meter.update(loss.item(), inputs.size(0))
            acc_meter.update(acc, inputs.size(0))
    
    return loss_meter.avg, acc_meter.avg

print("Validation function defined.")

## Section 7: Implement Validation Loop

Create a validation function to evaluate model performance on validation set and track accuracy and loss metrics.

In [ ]:
class AverageMeter:
    def __init__(self):
        self.reset()
    
    def reset(self):
        self.sum = 0.0
        self.count = 0
        self.avg = 0.0
    
    def update(self, value, n=1):
        self.sum += value * n
        self.count += n
        self.avg = self.sum / self.count if self.count != 0 else 0.0

def accuracy(output, target):
    _, preds = output.max(1)
    correct = preds.eq(target).sum().item()
    return 100.0 * correct / target.size(0)

def train_epoch(loader, model, criterion, optimizer, device):
    model.train()
    loss_meter = AverageMeter()
    acc_meter = AverageMeter()
    
    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Metrics
        acc = accuracy(outputs, targets)
        loss_meter.update(loss.item(), inputs.size(0))
        acc_meter.update(acc, inputs.size(0))
    
    return loss_meter.avg, acc_meter.avg

print("Training function defined.")

## Section 6: Implement Training Loop

Create a training function that performs forward pass, computes loss, backpropagation, and updates model weights with proper error handling.

In [ ]:
# Loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(
    model.parameters(),
    lr=learning_rate,
    momentum=momentum,
    weight_decay=weight_decay,
)

# Learning rate scheduler
scheduler = lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.1)

print("Loss function: CrossEntropyLoss")
print(f"Optimizer: SGD (lr={learning_rate}, momentum={momentum}, weight_decay={weight_decay})")
print(f"Scheduler: StepLR (step_size=20, gamma=0.1)")

## Section 5: Setup Training Components

Initialize the loss function (CrossEntropyLoss), optimizer (SGD or Adam), and learning rate scheduler for training.

In [ ]:
# Define ResNet-18 model adapted for CIFAR-100
def get_model(num_classes=100, pretrained=False):
    try:
        from torchvision.models import ResNet18_Weights
        weights = ResNet18_Weights.DEFAULT if pretrained else None
        model = models.resnet18(weights=weights)
    except ImportError:
        model = models.resnet18(pretrained=pretrained)
    
    # Adapt for CIFAR-100 (32x32 images)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

# Create model
model = get_model(num_classes=100, pretrained=False)
model = model.to(device)

# Print model info
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: ResNet-18")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## Section 4: Define Model Architecture

Define or load a convolutional neural network architecture suitable for CIFAR-100 classification with 100 classes.

In [ ]:
# Define transforms
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)),
])

# Load datasets
print("Loading CIFAR-100 dataset...")
train_dataset = datasets.CIFAR100(root="./data", train=True, download=True, transform=transform_train)
test_dataset = datasets.CIFAR100(root="./data", train=False, download=True, transform=transform_test)

# Create data loaders
pin_memory = device.type == "cuda"
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=pin_memory,
)
val_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin_memory,
)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

## Section 3: Load and Preprocess CIFAR-100 Dataset

Download CIFAR-100 dataset, apply data augmentation and normalization transforms, and create training and validation data loaders.

In [ ]:
# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Training parameters
epochs = 50
batch_size = 128
learning_rate = 0.1
momentum = 0.9
weight_decay = 5e-4
num_workers = 4

# Directory for saving checkpoints
checkpoint_dir = "./checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

## Section 2: Set Device Configuration

Configure the device (CPU or GPU) and set random seeds for reproducibility across training runs.

In [ ]:
import os
import warnings
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

# Suppress deprecation warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', message='.*dtype.*align should be passed as Python or NumPy boolean.*')

## Section 1: Import Required Libraries

Import necessary libraries including torch, torchvision, numpy, and matplotlib for model training and visualization.

# CIFAR-100 PyTorch Training Notebook

A complete training pipeline for the CIFAR-100 dataset using PyTorch with support for CPU and GPU training.